# Learn DreamLens `maximize()` step by step

This notebook uses the **real local DreamLens API**. It does not copy the optimizer loop and it does not use `from dreamlens import *`. Every public class is imported explicitly so you can see where each responsibility comes from.

The question we will ask is:

> What generated image makes one selected channel inside ResNet18 activate strongly?

Only the generated image parameters are optimized. The pretrained ResNet18 weights remain fixed.

## Latest executed result

This notebook was executed end to end with the parameter cell below. The generated image is linked from the notebook's saved output directory, so it is visible here immediately when the notebook is opened.

| Setting | Executed value |
| --- | ---: |
| Layer | `layer2.1.conv2` |
| Channel | `17` |
| Image size | `224 × 224` |
| Steps | `400` |
| Learning rate | `0.012` |
| Final transformed-view score | `39.1118` |
| Clean untransformed norm | `43.8359` |
| Clean size-normalized RMS | `1.5656` |
| Activation-map shape | `1 × 28 × 28` |

![Latest DreamLens maximize result](../learning_outputs/dreamlens_maximize_notebook/layer2_1_conv2_channel_17.png)

The cells below recreate this result and let you edit every important parameter.

## 1. Make the local `src/` package importable

In a normal installed project you would run `pip install -e .` once. This educational notebook also works directly from the repository by adding its local `src` directory to Python's import search path.

In [ ]:
from pathlib import Path
import sys

# The notebook may be started from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "dreamlens").is_dir():
    candidate = REPO_ROOT.parent
    if (candidate / "src" / "dreamlens").is_dir():
        REPO_ROOT = candidate
    else:
        raise RuntimeError("Run this notebook from the DreamLens repository.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_ROOT)
print("Importing DreamLens from:", SRC_DIR)

## 2. Explicit imports

The imports are intentionally explicit:

- `FeatureVisualizer` owns the optimization workflow.
- `FeatureTarget` describes the layer/channel question.
- `RenderConfig` stores image-optimization settings.
- `TransformConfig` stores rotation, scale, and translation settings.
- `OptimizationResult` is the object returned by `maximize()`.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from torchvision.models import ResNet18_Weights
from torchvision.models import resnet18

from dreamlens import FeatureTarget
from dreamlens import FeatureVisualizer
from dreamlens import OptimizationResult
from dreamlens import RenderConfig
from dreamlens import TransformConfig

print("PyTorch version:", torch.__version__)

## 3. Editable experiment parameters

This is the main cell to edit. The defaults use a tuned, stable ResNet18 channel-visualization configuration. Start by changing only `TARGET_CHANNEL`; keep the other values fixed until you understand their effects.

In [ ]:
# Reproducibility
SEED = 123                 # General setup seed
RENDER_SEED = 133          # Reset immediately before image generation
DEVICE = "cpu"            # Change to "cuda" if CUDA is available

# Target feature
TARGET_LAYER_NAME = "layer2.1.conv2"
TARGET_CHANNEL = 17
REDUCTION = "norm"        # Try: "norm", "mean", "sum", or "max"
TARGET_WEIGHT = 1.0
TARGET_SIGN = 1.0           # +1 maximizes; -1 suppresses the feature

# Generated image and optimizer
IMAGE_WIDTH = 224           # Standard ImageNet input size; more spatial detail
IMAGE_HEIGHT = 224
STEPS = 400                # High-quality setting; 500 gives only a small extra gain
LEARNING_RATE = 1.2e-2      # Tuned for the larger 224 x 224 canvas
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0

# Robustness transforms
ROTATE_DEGREES = 6         # Gentler than the original 10-degree transform
SCALE_MIN = 0.82
SCALE_MAX = 1.12
TRANSLATE_X = 0.01
TRANSLATE_Y = 0.01

# Output
OUTPUT_DIR = REPO_ROOT / "learning_outputs" / "dreamlens_maximize_notebook"
OUTPUT_FILENAME = f"{TARGET_LAYER_NAME.replace('.', '_')}_channel_{TARGET_CHANNEL}.png"

# Optional second experiment near the end of the notebook
RUN_BATCH_EXPERIMENT = False
BATCH_CHANNELS = [3, 17, 41, 64, 89, 121]

## 4. Seed every random generator

The initial Fourier coefficients and random transforms depend on random numbers. Using the same seed makes an experiment reproducible.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(DEVICE)
print("Using device:", device)

## 5. Load and freeze pretrained ResNet18

ResNet18 is the fixed judge. Gradients may pass backward through its operations to the generated image, but its weights are not optimized.

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)
model.eval()

for model_parameter in model.parameters():
    model_parameter.requires_grad_(False)

number_of_parameters = sum(parameter.numel() for parameter in model.parameters())
number_of_trainable_model_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print("Model parameters:", f"{number_of_parameters:,}")
print("Trainable model parameters:", number_of_trainable_model_parameters)

## 6. Resolve and inspect the target layer

`named_modules()` maps names such as `layer2.1.conv2` to real PyTorch module objects. DreamLens can accept either the name or the module; using the module here makes the selected object visible.

In [ ]:
modules_by_name = dict(model.named_modules())
if TARGET_LAYER_NAME not in modules_by_name:
    available_examples = [name for name in modules_by_name if "conv" in name][:20]
    raise KeyError(
        f"Unknown layer {TARGET_LAYER_NAME!r}. Example convolution layers: {available_examples}"
    )

target_layer = modules_by_name[TARGET_LAYER_NAME]
print("Target layer name:", TARGET_LAYER_NAME)
print("Target layer object:", target_layer)
print("Requested channel:", TARGET_CHANNEL)

## 7. Create the DreamLens controller

`FeatureVisualizer` owns the high-level loop: it creates an image canvas, installs a layer hook, runs the model, evaluates the objective, calls `backward()`, and asks AdamW to update the image parameters.

In [ ]:
visualizer = FeatureVisualizer(
    model=model,
    device=device,
    normalize=True,
    quiet=False,
)

print("Visualizer device:", visualizer.device)

## 8. Describe the target

This object contains the question, not the answer. With `channel=17` and `reduction='norm'`, DreamLens selects channel 17's complete spatial activation map and calculates its L2 norm. Internally the loss is the negative weighted score because AdamW minimizes.

In [ ]:
target = FeatureTarget(
    layer=target_layer,
    channel=TARGET_CHANNEL,
    reduction=REDUCTION,
    weight=TARGET_WEIGHT,
    sign=TARGET_SIGN,
)

print(target)

## 9. Configure transforms separately

A slightly different random view is used during each optimization step. This encourages the discovered pattern to remain useful after small rotations, resizes, and translations.

In [ ]:
transform_config = TransformConfig(
    rotate_degrees=ROTATE_DEGREES,
    scale_min=SCALE_MIN,
    scale_max=SCALE_MAX,
    translate_x=TRANSLATE_X,
    translate_y=TRANSLATE_Y,
)

print(transform_config)

## 10. Configure the image optimization

`RenderConfig.reference()` selects the reference Fourier canvas used by the successful project notebook. It creates random trainable Fourier coefficients. On every iteration, DreamLens applies frequency scaling and `torch.fft.irfft2()` **before** passing an ordinary reconstructed image to ResNet.

In [ ]:
render_config = RenderConfig.reference(
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    steps=STEPS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    transform=transform_config,
)

print(render_config)

## 11. Run `maximize()`

This is the line that starts optimization. We reset the random generators immediately before rendering so model construction cannot consume numbers intended for Fourier initialization. The call returns only after all configured steps have completed.

In [ ]:
# Reset immediately before rendering for reproducible Fourier initialization.
random.seed(RENDER_SEED)
np.random.seed(RENDER_SEED)
torch.manual_seed(RENDER_SEED)

result = visualizer.maximize(
    target=target,
    config=render_config,
)

assert isinstance(result, OptimizationResult)
print("Completed steps:", len(result.losses))
print("Final transformed-view loss:", result.losses[-1])
print("Final transformed-view score:", -result.losses[-1])

# Evaluate the final visible image once without a random transform.
clean_layer_output = visualizer.capture_layers(
    layers=[target_layer],
    input_tensor=result.as_nchw(),
    first_batch=False,
)[0]
clean_selected = clean_layer_output[:, TARGET_CHANNEL]

if REDUCTION == "norm":
    clean_score = torch.linalg.vector_norm(clean_selected)
elif REDUCTION == "mean":
    clean_score = clean_selected.mean()
elif REDUCTION == "sum":
    clean_score = clean_selected.sum()
elif REDUCTION == "max":
    clean_score = clean_selected.max()
else:
    raise ValueError(f"Unsupported notebook reduction: {REDUCTION}")

clean_rms_score = clean_score / clean_selected.numel() ** 0.5

print("Clean untransformed norm score:", clean_score.item())
print("Clean RMS score (size-normalized):", clean_rms_score.item())
print("Clean activation-map shape:", tuple(clean_selected.shape))

## 12. Display the generated RGB image

`result.as_hwc()` reconstructs the visible RGB image and returns height-width-color layout. ResNet received a normalized version during optimization; this method returns displayable values.

In [ ]:
generated_hwc = result.as_hwc().numpy()

print("Displayed image shape:", generated_hwc.shape)
print("Minimum RGB value:", generated_hwc.min())
print("Maximum RGB value:", generated_hwc.max())

plt.figure(figsize=(5, 5))
plt.imshow(np.clip(generated_hwc, 0.0, 1.0))
plt.title(f"{TARGET_LAYER_NAME} — channel {TARGET_CHANNEL}")
plt.axis("off")
plt.show()

## 13. Plot optimization history

For this one positive target, `score = -loss`. The curve can fluctuate because every training step uses a randomly transformed view. The clean score evaluates the finished image without a random transform. When comparing different image sizes, use the clean RMS score: the raw norm automatically grows when the activation map contains more spatial values.

In [ ]:
loss_history = np.asarray(result.losses, dtype=np.float32)
score_history = -loss_history

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(score_history) + 1), score_history)
plt.xlabel("Optimization step")
plt.ylabel("Target score (-loss)")
plt.title("DreamLens maximize progress")
plt.grid(alpha=0.3)
plt.show()

## 14. Save the result

The saved file is the visible RGB image, not the Fourier parameter tensor and not the normalized model input.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / OUTPUT_FILENAME
result.save(output_path)
print("Saved:", output_path)

## 15. Inspect where inverse FFT is used internally

This optional educational cell reads the local implementation. The helper is private, so normal application code should not call it directly. Notice that it creates complex coefficients, applies the fixed frequency scale, and calls `irfft2`. The model receives the resulting spatial image—not the Fourier coefficients.

In [ ]:
import inspect
import dreamlens.image_parameters as image_parameters_module

print(inspect.getsource(image_parameters_module._reference_fft_to_rgb))

## 16. Optional: render several channels in one batch

Set `RUN_BATCH_EXPERIMENT = True` in the parameter cell to enable this. `maximize_channels()` creates one independently optimized image per channel while running them as a batch.

In [ ]:
if RUN_BATCH_EXPERIMENT:
    random.seed(RENDER_SEED)
    np.random.seed(RENDER_SEED)
    torch.manual_seed(RENDER_SEED)

    batch_result = visualizer.maximize_channels(
        layer=target_layer,
        channels=BATCH_CHANNELS,
        reduction=REDUCTION,
        config=render_config,
    )

    columns = 3
    rows = int(np.ceil(len(BATCH_CHANNELS) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(12, 4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for index, channel in enumerate(BATCH_CHANNELS):
        image = batch_result.image[index].as_hwc().numpy()
        axes[index].imshow(np.clip(image, 0.0, 1.0))
        axes[index].set_title(f"channel {channel}")
        axes[index].axis("off")

    for axis in axes[len(BATCH_CHANNELS):]:
        axis.axis("off")

    figure.suptitle(TARGET_LAYER_NAME)
    figure.tight_layout()
    plt.show()
else:
    print("Batch experiment skipped. Set RUN_BATCH_EXPERIMENT=True to run it.")

## What to edit first

Recommended beginner experiments:

1. Change `TARGET_CHANNEL` from `17` to `3`, `41`, or `89`.
2. Compare `STEPS=42`, `160`, and the tuned `400`. More steps raise the score but eventually provide diminishing returns.
3. Set all transforms to identity (`ROTATE_DEGREES=0`, scales `1.0`, translations `0.0`) only as an experiment. It can greatly raise the raw score while producing fragile, high-frequency noise.
4. Compare `IMAGE_WIDTH=IMAGE_HEIGHT=160` with `224`. Compare their RMS scores, not only their raw norm scores.
5. Change `TARGET_LAYER_NAME` to `layer4.1.conv2` and select a valid channel below 512. Deeper-layer images are often more abstract.
6. Change `REDUCTION` from `norm` to `mean` and observe how the objective changes.

Remember the full flow:

```text
FeatureTarget + RenderConfig
→ FeatureVisualizer.maximize()
→ random trainable Fourier coefficients
→ frequency scaling
→ inverse FFT
→ RGB and normalization
→ random transform
→ frozen ResNet
→ captured target channel
→ negative score loss
→ backward()
→ AdamW updates Fourier coefficients
→ repeat
→ save visible RGB image
```